# Notebook 2 — Linear Regression with Evaluation Metrics

This notebook repeats the basic workflow and evaluates the test predictions
using MAE, MSE, RMSE, R² and MAPE.

## Use case

A bank wants to estimate the **balance after a transaction**. The target is
`Balance_After_Transaction`. The simplest model uses the transaction `Amount`
as its input.

Linear regression learns a straight-line relationship:

\[\text{Predicted Balance} = \text{Intercept} + (\text{Coefficient} \times \text{Amount})\]

> **Important:** Regression does not have classification accuracy. `model.score()`
returns **R²**, which measures how much variation in the target is explained by
the model. A high training score alone does not guarantee good performance on
new data.

## 1. Import libraries and read the data

The additional metric functions quantify different aspects of prediction
error. NumPy is used to calculate the square root for RMSE.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

df = pd.read_csv("banking_operations(1).csv")
print("Dataset shape:", df.shape)
df.head()

## 2. Select the feature and target

`Amount` is the input feature and `Balance_After_Transaction` is the continuous
target. This is a simple linear regression because there is one input feature.

In [ ]:
X = df[["Amount"]]
y = df["Balance_After_Transaction"]

X.describe()

## 3. Create the train–test split

The same fixed random seed is used so that results can be reproduced.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

## 4. Train the model

The model learns only from the training subset.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Learned equation: Balance = {model.intercept_:,.2f} + ({model.coef_[0]:.4f} × Amount)")

## 5. Predict one record

The new record must contain the same feature name and structure used during
training.

In [ ]:
single_record = pd.DataFrame({"Amount": [15000]})
single_prediction = model.predict(single_record)[0]

print(f"For an amount of ₹15,000, the predicted balance is ₹{single_prediction:,.2f}.")

## 6. Predict all test records

The error columns show both the signed difference and its absolute size.

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Amount": X_test["Amount"].values,
    "Actual": y_test.values,
    "Predicted": y_pred,
})
comparison["Error"] = comparison["Actual"] - comparison["Predicted"]
comparison["Absolute_Error"] = comparison["Error"].abs()
comparison.round(2)

## 7. Calculate different regression metrics

- **MAE (Mean Absolute Error):** Average absolute difference between actual and
  predicted balances. It is easy to understand because it uses balance units.
- **MSE (Mean Squared Error):** Average squared error. Large mistakes receive a
  stronger penalty, but the unit is squared.
- **RMSE (Root Mean Squared Error):** Square root of MSE. It returns to balance
  units while still penalizing large errors.
- **R² (Coefficient of Determination):** Proportion of target variation explained
  by the model. Higher is generally better, but it can be negative on test data.
- **MAPE (Mean Absolute Percentage Error):** Average absolute percentage error.
  It is intuitive, but becomes unstable when actual values are zero or near zero.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²", "MAPE"],
    "Value": [mae, mse, rmse, r2, mape],
    "Preferred Direction": ["Lower", "Lower", "Lower", "Higher", "Lower"],
})
metrics

## 8. Readable metric summary

No single metric tells the whole story. R² explains overall fit, MAE describes
the typical error, and RMSE highlights whether large errors are present.

In [ ]:
print(f"MAE : ₹{mae:,.2f}")
print(f"MSE : {mse:,.2f} squared balance units")
print(f"RMSE: ₹{rmse:,.2f}")
print(f"R²  : {r2:.4f} ({r2 * 100:.2f}% percentage-style display)")
print(f"MAPE: {mape:.2f}%")

## Conclusion

The metrics describe different dimensions of the same test predictions. For
this banking use case, MAE and RMSE are useful because they are expressed in
balance units, while R² indicates how much variation the model explains.

## Save trained model as pickle

Run this cell after training. The `.pkl` file is saved in the notebook’s current working directory. Only load pickle files from trusted sources.

In [ ]:
import pickle

with open("linear_regression_metrics.pkl", "wb") as file:
    pickle.dump(model, file)
print("Saved linear_regression_metrics.pkl")
